# StormFusion-MT Typhoon Retraining

This Google Colab notebook trains a replacement multi-task cyclone model using real IBTrACS track data and real ERA5 fields.

The model predicts six-hourly east/north storm motion, maximum wind, central pressure, radius of maximum wind, and 34/50/64-knot wind radii for four quadrants.

Important:
- This notebook requires a CDS API credential for ERA5. It never substitutes a zero atmospheric field.
- The default split is 1979-2015 training, 2016-2019 validation, and 2020 onward test.
- Windows are split by storm before training.
- Checkpoints, cached data, losses, predictions, and residuals are written to Google Drive.
- The model is an experimental research model, not an operational warning system.


In [ ]:
%pip -q install cdsapi xarray netCDF4 pandas numpy scipy scikit-learn matplotlib tqdm


In [ ]:
import os
import json
import math
import random
import time
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from google.colab import drive

drive.mount("/content/drive", force_remount=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DRIVE_ROOT = Path("/content/drive/MyDrive/typhoon_predict_stormfusion_mt")
DATA_ROOT = DRIVE_ROOT / "data"
FRAME_ROOT = DATA_ROOT / "era5_frames"
WINDOW_ROOT = DATA_ROOT / "windows"
CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
for folder in (DATA_ROOT, FRAME_ROOT, WINDOW_ROOT, CHECKPOINT_ROOT, ARTIFACT_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_name": "stormfusion_mt_v1",
    "track_url": "https://www.ncei.noaa.gov/data/international-best-track-archive-for-climate-stewardship-ibtracs/v04r01/access/csv/ibtracs.WP.list.v04r01.csv",
    "history_steps": 9,
    "history_hours": 48,
    "atmosphere_times": [-18, -12, -6, 0],
    "lead_hours": list(range(6, 121, 6)),
    "inner_channels": 26,
    "outer_channels": 14,
    "patch_size": 65,
    "inner_spacing_km": 25.0,
    "outer_spacing_km": 50.0,
    "max_storms": None,
    "era5_start_year": 1940,  # ERA5 reanalysis is unavailable before 1940
    "max_windows": 10000,
    "model_tier": "recommended",
    "batch_size": 8,
    "epochs": 60,
    "learning_rate": 1e-4,
    "weight_decay": 1e-2,
    "gradient_clip": 1.0,
    "patience": 10,
    "num_workers": 2,
    "use_bf16": True,
    "resume_checkpoint": "",
}
CONFIG_PATH = DRIVE_ROOT / "config.json"
CONFIG_PATH.write_text(json.dumps(CONFIG, indent=2))

assert torch.cuda.is_available(), "Select an NVIDIA A100 or H100 runtime before training."
DEVICE = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.version.cuda)
print("Drive run directory:", DRIVE_ROOT)


## ERA5 credentials

Create a Colab Secret named CDS_API_KEY containing your CDS API key in the format accepted by cdsapi if the ERA5 frame cache or processed window file is incomplete. The notebook also checks compatible cache locations from earlier runs and can use a hidden prompt as a fallback. A complete processed Drive cache can be reused without the key.

ERA5 is downloaded from the official Copernicus Climate Data Store. The request is storm-centered and cached on Drive, so restarting the runtime does not discard completed downloads.


In [ ]:
import getpass
import shutil
import cdsapi

CACHE_NAME = "stormfusion_windows.npz"
CACHED_WINDOWS = WINDOW_ROOT / CACHE_NAME
CACHE_ROOTS = [
    Path("/content/drive/MyDrive/typhoon_predict_stormfusion_mt/data/windows"),
    Path("/content/drive/MyDrive/typhoon_predict/data/windows"),
    Path("/content/drive/MyDrive/typhoon_predict_stormfusion/data/windows"),
]
# Include other top-level Drive project folders without scanning the entire Drive.
my_drive = Path("/content/drive/MyDrive")
if my_drive.exists():
    CACHE_ROOTS.extend(p / "data/windows" for p in my_drive.iterdir() if p.is_dir())

required_cache_keys = {
    "inner", "outer", "track", "env", "target", "target_mask",
    "storm_id", "year", "init_time", "init_lat", "init_lon",
}
def compatible_cache(path):
    try:
        with np.load(path, allow_pickle=True) as archive:
            return required_cache_keys.issubset(set(archive.files))
    except Exception:
        return False

cache_candidates = [root / CACHE_NAME for root in CACHE_ROOTS]
cache_source = next((path for path in cache_candidates if path.exists() and compatible_cache(path)), None)
if cache_source is not None:
    if cache_source != CACHED_WINDOWS:
        CACHED_WINDOWS.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(cache_source, CACHED_WINDOWS)
        print(f"Compatible processed cache copied from {cache_source.parent} into the current run directory.")
    else:
        print("Processed Drive window cache found; reusing it without a CDS credential.")
else:
    try:
        from google.colab import userdata
        CDS_API_KEY = userdata.get("CDS_API_KEY")
    except Exception:
        CDS_API_KEY = None
    if not CDS_API_KEY:
        CDS_API_KEY = os.environ.get("CDS_API_KEY")
    if not CDS_API_KEY:
        try:
            CDS_API_KEY = getpass.getpass("Enter CDS API key (hidden; leave blank to stop): ").strip()
        except (EOFError, KeyboardInterrupt):
            CDS_API_KEY = None
    if not CDS_API_KEY:
        raise RuntimeError(
            "No compatible processed Drive cache was found and CDS_API_KEY is unavailable. "
            "Add a Colab Secret named CDS_API_KEY or enter it at the hidden prompt."
        )
    Path.home().joinpath(".cdsapirc").write_text(
        "url: https://cds.climate.copernicus.eu/api\n"
        f"key: {CDS_API_KEY}\n"
    )
    print("CDS API configuration written without printing the credential.")


## Download and normalize IBTrACS

The track table keeps the JTWC/USA convention when available. Wind radii are preserved separately by threshold and quadrant. Missing values stay missing and are represented by masks during training.


In [ ]:
TRACK_FILE = DATA_ROOT / "ibtracs_wp_v04r01.csv"
if not TRACK_FILE.exists():
    response = requests.get(CONFIG["track_url"], timeout=180)
    response.raise_for_status()
    TRACK_FILE.write_bytes(response.content)

raw_track = pd.read_csv(TRACK_FILE, skiprows=[1], low_memory=False)
print("Raw shape:", raw_track.shape)
print("Relevant columns:", [c for c in raw_track.columns if c.startswith(("USA_", "WMO_"))][:50])

def first_existing(frame, names, required=True):
    for name in names:
        if name in frame.columns:
            return name
    if required:
        raise KeyError(f"None of these columns exist: {names}")
    return None

def numeric(series):
    value = pd.to_numeric(series, errors="coerce").astype("float32")
    value = value.mask(value < -900)
    value = value.mask(value > 1e6)
    return value

sid_col = first_existing(raw_track, ["SID", "SERIAL_NUM"])
time_col = first_existing(raw_track, ["ISO_TIME"])
lat_col = first_existing(raw_track, ["LAT"])
lon_col = first_existing(raw_track, ["LON"])
wind_col = first_existing(raw_track, ["USA_WIND", "WMO_WIND"], required=False)
pres_col = first_existing(raw_track, ["USA_PRES", "WMO_PRES"], required=False)
gust_col = first_existing(raw_track, ["USA_GUST", "WMO_GUST"], required=False)
rmw_col = first_existing(raw_track, ["USA_RMW", "WMO_RMW"], required=False)

track = pd.DataFrame({
    "sid": raw_track[sid_col].astype(str),
    "name": raw_track[first_existing(raw_track, ["NAME"], required=False)].astype(str) if "NAME" in raw_track else "",
    "basin": raw_track[first_existing(raw_track, ["BASIN"], required=False)].astype(str) if "BASIN" in raw_track else "WP",
    "time": pd.to_datetime(raw_track[time_col], errors="coerce", utc=True),
    "lat": numeric(raw_track[lat_col]),
    "lon": numeric(raw_track[lon_col]),
    "vmax": numeric(raw_track[wind_col]) if wind_col else np.nan,
    "pressure": numeric(raw_track[pres_col]) if pres_col else np.nan,
    "gust": numeric(raw_track[gust_col]) if gust_col else np.nan,
    "rmw": numeric(raw_track[rmw_col]) if rmw_col else np.nan,
})

for threshold in (34, 50, 64):
    for quadrant in ("ne", "se", "sw", "nw"):
        candidates = [
            f"USA_R{threshold}_{quadrant.upper()}",
            f"WMO_R{threshold}_{quadrant.upper()}",
        ]
        source = first_existing(raw_track, candidates, required=False)
        track[f"r{threshold}_{quadrant}"] = numeric(raw_track[source]) if source else np.nan

track = track[(track["basin"] == "WP") & track["time"].notna() & track["lat"].notna() & track["lon"].notna()].copy()
track["year"] = track["time"].dt.year.astype(int)
era5_start_year = CONFIG.get("era5_start_year")
if era5_start_year is not None:
    before = track["sid"].nunique()
    track = track[track["year"] >= int(era5_start_year)].copy()
    print(f"Dropped storms before {era5_start_year}: kept {track['sid'].nunique()} of {before}.")
track["lon"] = ((track["lon"] + 180.0) % 360.0) - 180.0
track = track.sort_values(["sid", "time"]).reset_index(drop=True)

if CONFIG["max_storms"] is not None:
    keep = track["sid"].drop_duplicates().iloc[:CONFIG["max_storms"]]
    track = track[track["sid"].isin(keep)].copy()

print("Normalized rows:", len(track))
print("Storms:", track["sid"].nunique())
print("Years:", int(track["year"].min()), "to", int(track["year"].max()))
print("Radius availability:")
print(track[[c for c in track.columns if c.startswith("r")]].notna().mean().round(3))


## ERA5 frame download and storm-centered patch extraction

The inner patch contains wind, moisture, temperature, geopotential, pressure, SST, near-surface wind, land-sea mask, and surface geopotential.

The outer patch provides larger-scale steering context. Four analysis times are used: t-18, t-12, t-6, and t0.

The implementation uses real ERA5 values. A missing or failed frame raises an error rather than creating a zero patch.


In [ ]:
import xarray as xr

PRESSURE_LEVELS = ["925", "850", "700", "500", "200"]
PRESSURE_VARIABLES = [
    "u_component_of_wind",
    "v_component_of_wind",
    "specific_humidity",
    "temperature",
    "geopotential",
]
SINGLE_VARIABLES = [
    "10m_u_component_of_wind",
    "10m_v_component_of_wind",
    "mean_sea_level_pressure",
    "sea_surface_temperature",
    "surface_pressure",
    "land_sea_mask",
    "geopotential",
]

VAR_ALIASES = {
    "u_component_of_wind": ["u_component_of_wind", "u"],
    "v_component_of_wind": ["v_component_of_wind", "v"],
    "specific_humidity": ["specific_humidity", "q"],
    "temperature": ["temperature", "t"],
    "geopotential": ["geopotential", "z"],
    "10m_u_component_of_wind": ["10m_u_component_of_wind", "u10"],
    "10m_v_component_of_wind": ["10m_v_component_of_wind", "v10"],
    "mean_sea_level_pressure": ["mean_sea_level_pressure", "msl"],
    "sea_surface_temperature": ["sea_surface_temperature", "sst"],
    "surface_pressure": ["surface_pressure", "sp"],
    "land_sea_mask": ["land_sea_mask", "lsm"],
}

def frame_key(ts, lat, lon):
    ts = pd.Timestamp(ts).tz_convert("UTC")
    return f"{ts:%Y%m%d%H}_{float(lat):+.2f}_{float(lon):+.2f}".replace("-", "m").replace("+", "p").replace(".", "d")

def request_era5_frame(ts, lat, lon):
    ts = pd.Timestamp(ts).tz_convert("UTC")
    key = frame_key(ts, lat, lon)
    pressure_path = FRAME_ROOT / f"{key}_pressure.nc"
    single_path = FRAME_ROOT / f"{key}_single.nc"
    if pressure_path.exists() and single_path.exists():
        return pressure_path, single_path

    client = cdsapi.Client(quiet=True)
    lon0 = float(lon) % 360.0
    west = max(0.0, lon0 - 22.0)
    east = min(360.0, lon0 + 22.0)
    north = min(90.0, float(lat) + 22.0)
    south = max(-90.0, float(lat) - 22.0)
    common = {
        "product_type": ["reanalysis"],
        "year": [f"{ts.year:04d}"],
        "month": [f"{ts.month:02d}"],
        "day": [f"{ts.day:02d}"],
        "time": [f"{ts.hour:02d}:00"],
        "area": [north, west, south, east],
        "data_format": "netcdf",
        "download_format": "unarchived",
    }
    pressure_request = {
        **common,
        "variable": PRESSURE_VARIABLES,
        "pressure_level": PRESSURE_LEVELS,
    }
    single_request = {
        **common,
        "variable": SINGLE_VARIABLES,
    }
    if not pressure_path.exists():
        client.retrieve("reanalysis-era5-pressure-levels", pressure_request, str(pressure_path))
    if not single_path.exists():
        client.retrieve("reanalysis-era5-single-levels", single_request, str(single_path))
    if not pressure_path.exists() or not single_path.exists():
        raise RuntimeError(f"ERA5 request did not create both files for {ts}")
    return pressure_path, single_path

def _coord_name(ds, candidates):
    for name in candidates:
        if name in ds.coords or name in ds.dims:
            return name
    raise KeyError(f"Could not find coordinate among {candidates}; found {list(ds.coords)}")

def _select_dataarray(ds, variable, level=None):
    candidates = VAR_ALIASES.get(variable, [variable])
    found = next((name for name in candidates if name in ds), None)
    if found is None:
        raise KeyError(f"ERA5 variable {variable} missing (tried {candidates}); found {list(ds.data_vars)}")
    da = ds[found]
    for time_dim in ("time", "valid_time"):
        if time_dim in da.dims:
            da = da.isel({time_dim: 0})
    if level is not None:
        level_dim = _coord_name(ds, ["pressure_level", "isobaricInhPa", "level"])
        da = da.sel({level_dim: int(level)}, method="nearest")
    return da

def _interp_patch(da, lat_values, lon_values):
    lat_name = _coord_name(da.to_dataset(name="x"), ["latitude", "lat"])
    lon_name = _coord_name(da.to_dataset(name="x"), ["longitude", "lon"])
    da = da.sortby(lat_name)
    lon_coord = da[lon_name]
    if float(lon_coord.max()) > 180.0:
        left = da.assign_coords({lon_name: lon_coord - 360.0})
        right = da.assign_coords({lon_name: lon_coord + 360.0})
        da = xr.concat([left, da, right], dim=lon_name).sortby(lon_name)
        lon_values = ((np.asarray(lon_values) + 180.0) % 360.0) - 180.0
    lat_da = xr.DataArray(np.asarray(lat_values), dims="y")
    lon_da = xr.DataArray(np.asarray(lon_values), dims="x")
    out = da.interp({lat_name: lat_da, lon_name: lon_da}, method="linear")
    return np.asarray(out.values, dtype="float32")

def _local_grid(lat, lon, spacing_km, size):
    offsets = (np.arange(size, dtype="float32") - (size - 1) / 2.0) * spacing_km
    lat_values = float(lat) + offsets / 111.2
    cos_lat = max(0.2, math.cos(math.radians(float(lat))))
    lon_values = float(lon) + offsets / (111.2 * cos_lat)
    return lat_values, lon_values

def extract_frame_patches(ts, lat, lon):
    pressure_path, single_path = request_era5_frame(ts, lat, lon)
    with xr.open_dataset(pressure_path) as pressure_ds, xr.open_dataset(single_path) as single_ds:
        inner_lat, inner_lon = _local_grid(lat, lon, CONFIG["inner_spacing_km"], CONFIG["patch_size"])
        outer_lat, outer_lon = _local_grid(lat, lon, CONFIG["outer_spacing_km"], CONFIG["patch_size"])

        def p(var, level, lat_values, lon_values):
            return _interp_patch(_select_dataarray(pressure_ds, var, level), lat_values, lon_values)

        def s(var, lat_values, lon_values):
            return _interp_patch(_select_dataarray(single_ds, var), lat_values, lon_values)

        inner_parts = []
        for level in ("925", "850", "700", "500", "200"):
            inner_parts += [p("u_component_of_wind", level, inner_lat, inner_lon)]
            inner_parts += [p("v_component_of_wind", level, inner_lat, inner_lon)]
        for level in ("850", "700", "500"):
            inner_parts.append(p("specific_humidity", level, inner_lat, inner_lon))
        for level in ("850", "500", "200"):
            inner_parts.append(p("temperature", level, inner_lat, inner_lon))
        for level in ("850", "500", "200"):
            inner_parts.append(p("geopotential", level, inner_lat, inner_lon))
        inner_parts += [
            s("mean_sea_level_pressure", inner_lat, inner_lon),
            s("surface_pressure", inner_lat, inner_lon),
            s("sea_surface_temperature", inner_lat, inner_lon),
            s("10m_u_component_of_wind", inner_lat, inner_lon),
            s("10m_v_component_of_wind", inner_lat, inner_lon),
            s("land_sea_mask", inner_lat, inner_lon),
            s("geopotential", inner_lat, inner_lon),
        ]

        outer_parts = []
        for level in ("850", "700", "500", "200"):
            outer_parts += [p("u_component_of_wind", level, outer_lat, outer_lon)]
            outer_parts += [p("v_component_of_wind", level, outer_lat, outer_lon)]
        for level in ("850", "500", "200"):
            outer_parts.append(p("geopotential", level, outer_lat, outer_lon))
        outer_parts += [
            p("specific_humidity", "700", outer_lat, outer_lon),
            s("mean_sea_level_pressure", outer_lat, outer_lon),
            s("sea_surface_temperature", outer_lat, outer_lon),
        ]

    inner = np.stack(inner_parts).astype("float32")
    outer = np.stack(outer_parts).astype("float32")
    if inner.shape != (26, CONFIG["patch_size"], CONFIG["patch_size"]):
        raise ValueError(f"Inner patch shape mismatch: {inner.shape}")
    if outer.shape != (14, CONFIG["patch_size"], CONFIG["patch_size"]):
        raise ValueError(f"Outer patch shape mismatch: {outer.shape}")
    return inner, outer

def environmental_summary(inner):
    mean = lambda x: float(np.nanmean(x))
    return np.asarray([
        mean(inner[8] - inner[2]),
        mean(inner[9] - inner[3]),
        mean(inner[2]),
        mean(inner[3]),
        mean(inner[11]),
        mean(inner[13]),
        mean(inner[19]),
        mean(inner[21]),
        mean(inner[24]),
        mean(inner[25]),
    ], dtype="float32")

print("ERA5 functions ready. Frame downloads will be cached on Drive.")


## Build leakage-safe training windows

Each sample contains:
- 9 six-hourly historical storm states from t-48h through t0
- 4 ERA5 times from t-18h through t0
- 20 six-hourly future states from 6h through 120h
- 40 track features, including availability masks
- 17 output variables per lead

Radii use the JTWC/USA fields when available. Missing structure labels are masked rather than filled as physical zeros.


In [ ]:
RADIUS_NAMES = [f"r{threshold}_{quadrant}" for threshold in (34, 50, 64) for quadrant in ("ne", "se", "sw", "nw")]
STATE_NAMES = [
    "east_step_km", "north_step_km", "vmax_kt", "pressure_hpa", "rmw_km",
    *RADIUS_NAMES,
]
STATE_SCALE = np.asarray([100.0, 100.0, 35.0, 20.0, 50.0] + [50.0] * 12, dtype="float32")
assert len(STATE_NAMES) == 17

def valid_number(value):
    return value is not None and np.isfinite(value)

def local_motion_km(lat0, lon0, lat1, lon1):
    dlat = float(lat1) - float(lat0)
    dlon = ((float(lon1) - float(lon0) + 180.0) % 360.0) - 180.0
    north = dlat * 111.2
    east = dlon * 111.2 * math.cos(math.radians((float(lat0) + float(lat1)) / 2.0))
    return east, north

def nearest_row(group, target_time, tolerance_hours=1.5):
    delta = (group["time"] - target_time).abs()
    index = delta.idxmin()
    if pd.isna(index) or delta.loc[index] > pd.Timedelta(hours=tolerance_hours):
        return None
    return group.loc[index]

def history_features(history_rows, base_row, t0):
    sequence = np.zeros((len(history_rows), 40), dtype="float32")
    previous = None
    for i, row in enumerate(history_rows):
        east, north = local_motion_km(base_row.lat, base_row.lon, row.lat, row.lon)
        if previous is None:
            step_east, step_north = 0.0, 0.0
        else:
            step_east, step_north = local_motion_km(previous.lat, previous.lon, row.lat, row.lon)
        features = sequence[i]
        features[0:4] = [east, north, step_east, step_north]
        values = [row.vmax, row.pressure, row.gust, row.rmw]
        features[4] = float(values[0]) if valid_number(values[0]) else 0.0
        features[5] = float(values[1]) if valid_number(values[1]) else 0.0
        features[6] = float(values[2]) if valid_number(values[2]) else 0.0
        features[7] = float(values[3]) if valid_number(values[3]) else 0.0
        for j, name in enumerate(RADIUS_NAMES):
            value = row[name]
            features[8 + j] = float(value) if valid_number(value) else 0.0
        features[20] = 0.0
        phase = 2.0 * math.pi * t0.dayofyear / 365.25
        features[21:23] = [math.sin(phase), math.cos(phase)]
        features[23] = float((t0 - row.time).total_seconds() / 3600.0)
        fields = [row.vmax, row.pressure, row.gust, row.rmw] + [row[name] for name in RADIUS_NAMES]
        features[24] = float(valid_number(fields[0]))
        features[25] = float(valid_number(fields[1]))
        features[26] = float(valid_number(fields[2]))
        features[27] = float(valid_number(fields[3]))
        features[28:40] = [float(valid_number(value)) for value in fields[4:]]
        previous = row
    return sequence

def future_targets(future_rows, base_row):
    target = np.zeros((len(future_rows), 17), dtype="float32")
    mask = np.zeros((len(future_rows), 17), dtype=bool)
    previous = base_row
    for i, row in enumerate(future_rows):
        east, north = local_motion_km(previous.lat, previous.lon, row.lat, row.lon)
        target[i, 0:2] = [east, north]
        mask[i, 0:2] = True
        for j, name in enumerate(["vmax", "pressure", "rmw"] + RADIUS_NAMES, start=2):
            value = row[name]
            if valid_number(value):
                target[i, j] = float(value)
                mask[i, j] = True
        previous = row
    return target, mask

patch_cache = {}
def cached_patch(row):
    key = frame_key(row.time, row.lat, row.lon)
    if key not in patch_cache:
        patch_cache[key] = extract_frame_patches(row.time, row.lat, row.lon)
    return patch_cache[key]

def build_windows(track_frame):
    inner_windows, outer_windows, track_windows, env_windows = [], [], [], []
    target_windows, mask_windows = [], []
    ids, years, init_times, init_lats, init_lons = [], [], [], [], []
    history_offsets = [pd.Timedelta(hours=-6 * i) for i in range(CONFIG["history_steps"] - 1, -1, -1)]
    lead_offsets = [pd.Timedelta(hours=h) for h in CONFIG["lead_hours"]]

    groups = list(track_frame.groupby("sid", sort=False))
    order = np.random.RandomState(42).permutation(len(groups))
    groups = [groups[i] for i in order]
    if CONFIG["max_storms"] is not None:
        groups = groups[:CONFIG["max_storms"]]

    for sid, group in tqdm(groups, desc="Building storm windows"):
        group = group.sort_values("time")
        candidate_times = group["time"].drop_duplicates().tolist()
        for t0 in candidate_times:
            base = nearest_row(group, t0)
            if base is None:
                continue
            history_rows = [nearest_row(group, t0 + offset) for offset in history_offsets]
            future_rows = [nearest_row(group, t0 + offset) for offset in lead_offsets]
            if any(row is None for row in history_rows + future_rows):
                continue

            atmosphere_rows = [nearest_row(group, t0 + pd.Timedelta(hours=h)) for h in CONFIG["atmosphere_times"]]
            if any(row is None for row in atmosphere_rows):
                continue

            try:
                patches = [cached_patch(row) for row in atmosphere_rows]
            except Exception as error:
                print(f"Skipping {sid} {t0} because ERA5 extraction failed: {error}")
                continue

            inner = np.stack([pair[0] for pair in patches])
            outer = np.stack([pair[1] for pair in patches])
            env = np.stack([environmental_summary(pair[0]) for pair in patches])
            hist = history_features(history_rows, base, t0)
            target, target_mask = future_targets(future_rows, base)

            inner_windows.append(inner)
            outer_windows.append(outer)
            track_windows.append(hist)
            env_windows.append(env)
            target_windows.append(target)
            mask_windows.append(target_mask)
            ids.append(str(sid))
            years.append(int(t0.year))
            init_times.append(str(t0))
            init_lats.append(float(base.lat))
            init_lons.append(float(base.lon))

            if len(inner_windows) >= CONFIG["max_windows"]:
                break
        if len(inner_windows) >= CONFIG["max_windows"]:
            break

    if not inner_windows:
        raise RuntimeError("No windows were built. Check track cadence, dates, ERA5 credentials, and cache.")
    arrays = {
        "inner": np.asarray(inner_windows, dtype="float32"),
        "outer": np.asarray(outer_windows, dtype="float32"),
        "track": np.asarray(track_windows, dtype="float32"),
        "env": np.asarray(env_windows, dtype="float32"),
        "target": np.asarray(target_windows, dtype="float32"),
        "target_mask": np.asarray(mask_windows, dtype="bool"),
        "storm_id": np.asarray(ids),
        "year": np.asarray(years, dtype="int16"),
        "init_time": np.asarray(init_times),
        "init_lat": np.asarray(init_lats, dtype="float32"),
        "init_lon": np.asarray(init_lons, dtype="float32"),
    }
    return arrays

WINDOW_FILE = WINDOW_ROOT / "stormfusion_windows.npz"
if not WINDOW_FILE.exists():
    window_arrays = build_windows(track)
    np.savez_compressed(WINDOW_FILE, **window_arrays)
else:
    window_arrays = dict(np.load(WINDOW_FILE, allow_pickle=True))

for key, value in window_arrays.items():
    print(key, value.shape, value.dtype)


## Build the StormFusion-MT model

The recommended model uses:
- Separate inner and outer atmospheric encoders
- Track and environmental token encoders
- Temporal Transformer context
- Learned lead-time queries
- A cross-attention decoder
- A multi-task output head

The model is intentionally moderate in size. It is not a full GraphCast or GenCast replacement.


In [ ]:
class FrameEncoder(nn.Module):
    def __init__(self, in_channels, d_model, compact=False):
        super().__init__()
        widths = (32, 64, d_model) if compact else (64, 128, d_model)
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, widths[0], 3, padding=1),
            nn.GroupNorm(8, widths[0]),
            nn.GELU(),
            nn.Conv2d(widths[0], widths[1], 3, stride=2, padding=1),
            nn.GroupNorm(8, widths[1]),
            nn.GELU(),
            nn.Conv2d(widths[1], widths[2], 3, stride=2, padding=1),
            nn.GroupNorm(8, widths[2]),
            nn.GELU(),
            nn.AdaptiveAvgPool2d(1),
        )

    def forward(self, x):
        batch, steps, channels, height, width = x.shape
        z = self.net(x.reshape(batch * steps, channels, height, width))
        return z.flatten(1).reshape(batch, steps, -1)

class StormFusionMT(nn.Module):
    def __init__(self, tier="recommended", lead_count=20):
        super().__init__()
        compact = tier == "compact"
        d_model = 128 if compact else 192
        heads = 4 if compact else 6
        ffn = 256 if compact else 512
        context_depth = 1 if compact else 2
        decoder_depth = 2 if compact else 4

        self.d_model = d_model
        self.lead_count = lead_count
        self.inner_encoder = FrameEncoder(26, d_model, compact=compact)
        self.outer_encoder = FrameEncoder(14, d_model, compact=compact)
        self.track_proj = nn.Linear(40, d_model)
        self.env_proj = nn.Linear(10, d_model)
        self.inner_time = nn.Parameter(torch.zeros(1, 4, d_model))
        self.outer_time = nn.Parameter(torch.zeros(1, 4, d_model))
        self.track_time = nn.Parameter(torch.zeros(1, 9, d_model))
        self.env_time = nn.Parameter(torch.zeros(1, 4, d_model))

        context_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=ffn,
            dropout=0.1, batch_first=True, norm_first=True, activation="gelu"
        )
        self.context = nn.TransformerEncoder(context_layer, num_layers=context_depth)

        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=ffn,
            dropout=0.1, batch_first=True, norm_first=True, activation="gelu"
        )
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers=decoder_depth)
        self.lead_queries = nn.Parameter(torch.randn(1, lead_count, d_model) * 0.02)
        self.state_head = nn.Linear(d_model, 17)
        self.log_scale_head = nn.Linear(d_model, 17)

    def forward(self, inner, outer, track, env):
        inner_tokens = self.inner_encoder(inner) + self.inner_time
        outer_tokens = self.outer_encoder(outer) + self.outer_time
        track_tokens = self.track_proj(track) + self.track_time
        env_tokens = self.env_proj(env) + self.env_time
        memory = torch.cat([inner_tokens, outer_tokens, track_tokens, env_tokens], dim=1)
        memory = self.context(memory)
        queries = self.lead_queries.expand(inner.shape[0], -1, -1)
        decoded = self.decoder(queries, memory)
        state = self.state_head(decoded)
        log_scale = self.log_scale_head(decoded).clamp(-5.0, 3.0)
        return state, log_scale

model = StormFusionMT(CONFIG["model_tier"], lead_count=len(CONFIG["lead_hours"])).to(DEVICE)
print("Trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))
print("Model tier:", CONFIG["model_tier"])


## Dataset, normalization, and loss

The target scale follows the research report: movement in kilometers, wind in knots, pressure in hPa, and radius in kilometers.

The target mask prevents missing RMW and radius labels from becoming fake zero observations. The physical consistency penalty discourages negative wind/radius values and incorrectly nested wind radii.


In [ ]:
class WindowDataset(Dataset):
    def __init__(self, arrays, indices):
        self.arrays = arrays
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        i = int(self.indices[item])
        return {
            "inner": torch.from_numpy(self.arrays["inner"][i]),
            "outer": torch.from_numpy(self.arrays["outer"][i]),
            "track": torch.from_numpy(self.arrays["track"][i]),
            "env": torch.from_numpy(self.arrays["env"][i]),
            "target": torch.from_numpy(self.arrays["target"][i]),
            "target_mask": torch.from_numpy(self.arrays["target_mask"][i].astype("float32")),
        }

def storm_year_split(arrays):
    years = arrays["year"].astype(int)
    storm_ids = arrays["storm_id"].astype(str)
    first_year = {}
    for storm in np.unique(storm_ids):
        first_year[storm] = int(years[storm_ids == storm].min())
    train_storms = {storm for storm, year in first_year.items() if year <= 2015}
    valid_storms = {storm for storm, year in first_year.items() if 2016 <= year <= 2019}
    test_storms = {storm for storm, year in first_year.items() if year >= 2020}
    train = np.where(np.isin(storm_ids, list(train_storms)))[0]
    valid = np.where(np.isin(storm_ids, list(valid_storms)))[0]
    test = np.where(np.isin(storm_ids, list(test_storms)))[0]
    if len(train) == 0 or len(valid) == 0:
        raise RuntimeError(
            f"Empty split: train={len(train)}, validation={len(valid)}, test={len(test)}. "
            "Download more years or adjust the split explicitly."
        )
    print("Storm split counts:", len(train_storms), len(valid_storms), len(test_storms))
    return train, valid, test

train_idx, valid_idx, test_idx = storm_year_split(window_arrays)
train_loader = DataLoader(
    WindowDataset(window_arrays, train_idx),
    batch_size=CONFIG["batch_size"], shuffle=True,
    num_workers=CONFIG["num_workers"], pin_memory=True, persistent_workers=CONFIG["num_workers"] > 0
)
valid_loader = DataLoader(
    WindowDataset(window_arrays, valid_idx),
    batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=True, persistent_workers=CONFIG["num_workers"] > 0
)
test_loader = DataLoader(
    WindowDataset(window_arrays, test_idx),
    batch_size=CONFIG["batch_size"], shuffle=False,
    num_workers=CONFIG["num_workers"], pin_memory=True, persistent_workers=CONFIG["num_workers"] > 0
)

TARGET_SCALE = torch.tensor(STATE_SCALE, device=DEVICE).view(1, 1, -1)

def masked_huber(prediction, target, mask):
    loss = F.smooth_l1_loss(prediction, target, reduction="none")
    return (loss * mask).sum() / mask.sum().clamp_min(1.0)

def masked_gaussian_nll(prediction, log_scale, target, mask):
    scale = log_scale.exp().clamp_min(1e-3)
    nll = 0.5 * ((prediction - target) / scale).pow(2) + log_scale
    return (nll * mask).sum() / mask.sum().clamp_min(1.0)

def physical_penalty(prediction):
    physical = prediction * TARGET_SCALE
    vmax = physical[..., 2]
    rmw = physical[..., 4]
    r34 = physical[..., 5:9]
    r50 = physical[..., 9:13]
    r64 = physical[..., 13:17]
    nonnegative = F.relu(-vmax).mean() + F.relu(-rmw).mean()
    nonnegative = nonnegative + F.relu(-r34).mean() + F.relu(-r50).mean() + F.relu(-r64).mean()
    nesting = F.relu(r50 - r34).mean() + F.relu(r64 - r50).mean()
    return 0.01 * (nonnegative + nesting)

def total_loss(prediction, log_scale, target, mask):
    normalized_target = target / TARGET_SCALE
    huber_loss = masked_huber(prediction, normalized_target, mask)
    nll_loss = masked_gaussian_nll(prediction, log_scale, normalized_target, mask)
    consistency = physical_penalty(prediction)
    total = 0.7 * huber_loss + 0.3 * nll_loss + consistency
    return total, {
        "data_loss": float(huber_loss.detach().cpu()),
        "nll_loss": float(nll_loss.detach().cpu()),
        "physical_penalty": float(consistency.detach().cpu()),
    }

print("Train/validation/test windows:", len(train_idx), len(valid_idx), len(test_idx))
print("Target variables:", STATE_NAMES)


## Train with BF16, checkpoints, and automatic Drive saving

Every epoch writes:
- last.pt
- best.pt
- history.json
- config.json
- loss_curve.png
- validation predictions after the best epoch

The current checkpoint is not loaded because its CNN-GRU architecture and target definition are incompatible with StormFusion-MT. Use RESUME_CHECKPOINT only for a StormFusion-MT checkpoint created by this notebook.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG["epochs"])
use_amp = bool(CONFIG["use_bf16"] and DEVICE.type == "cuda")
amp_dtype = torch.bfloat16

start_epoch = 0
best_val = float("inf")
bad_epochs = 0
history = []

resume = CONFIG.get("resume_checkpoint", "")
if resume:
    checkpoint = torch.load(resume, map_location=DEVICE)
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    scheduler.load_state_dict(checkpoint["scheduler"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_val = float(checkpoint["best_val"])
    history = checkpoint.get("history", [])
    print("Resumed from", resume, "at epoch", start_epoch)

def move_batch(batch):
    return {key: value.to(DEVICE, non_blocking=True) for key, value in batch.items()}

def run_epoch(loader, training):
    model.train(training)
    total = 0.0
    details = {"data_loss": 0.0, "nll_loss": 0.0, "physical_penalty": 0.0}
    count = 0
    for batch in loader:
        batch = move_batch(batch)
        if training:
            optimizer.zero_grad(set_to_none=True)
        context = torch.cuda.amp.autocast(dtype=amp_dtype, enabled=use_amp) if DEVICE.type == "cuda" else nullcontext()
        grad_context = torch.enable_grad() if training else torch.no_grad()
        with grad_context, context:
            prediction, log_scale = model(batch["inner"], batch["outer"], batch["track"], batch["env"])
            loss, parts = total_loss(prediction, log_scale, batch["target"], batch["target_mask"])
        if training:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
            optimizer.step()
        batch_size = batch["inner"].shape[0]
        total += float(loss.detach().cpu()) * batch_size
        count += batch_size
        for key in details:
            details[key] += parts[key] * batch_size
    result = {"loss": total / max(1, count)}
    result.update({key: value / max(1, count) for key, value in details.items()})
    return result

for epoch in range(start_epoch, CONFIG["epochs"]):
    train_metrics = run_epoch(train_loader, training=True)
    valid_metrics = run_epoch(valid_loader, training=False)
    scheduler.step()
    row = {
        "epoch": epoch,
        "lr": scheduler.get_last_lr()[0],
        "train": train_metrics,
        "validation": valid_metrics,
    }
    history.append(row)
    print(
        f"epoch {epoch:03d} | train {train_metrics['loss']:.5f} | "
        f"val {valid_metrics['loss']:.5f} | lr {row['lr']:.3e}"
    )

    payload = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val": best_val,
        "history": history,
        "config": CONFIG,
        "state_names": STATE_NAMES,
        "state_scale": STATE_SCALE,
    }
    torch.save(payload, CHECKPOINT_ROOT / "last.pt")
    (CHECKPOINT_ROOT / "history.json").write_text(json.dumps(history, indent=2))
    (CHECKPOINT_ROOT / "config.json").write_text(json.dumps(CONFIG, indent=2))

    if valid_metrics["loss"] < best_val:
        best_val = valid_metrics["loss"]
        bad_epochs = 0
        payload["best_val"] = best_val
        torch.save(payload, CHECKPOINT_ROOT / "best.pt")
    else:
        bad_epochs += 1
        if bad_epochs >= CONFIG["patience"]:
            print("Early stopping at epoch", epoch)
            break

plt.figure(figsize=(8, 4))
plt.plot([x["train"]["loss"] for x in history], label="train")
plt.plot([x["validation"]["loss"] for x in history], label="validation")
plt.xlabel("epoch")
plt.ylabel("masked normalized loss")
plt.legend()
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(ARTIFACT_ROOT / "loss_curve.png", dpi=160)
plt.show()
print("Best checkpoint:", CHECKPOINT_ROOT / "best.pt")


## Validation metrics and covariance residual export

This evaluates the deterministic mean prediction. It also saves complete track/wind/pressure residuals for the generalized covariance experiment.

The residual matrix is only valid for covariance estimation after the model has been evaluated on independent held-out storm cases.


In [ ]:
best_payload = torch.load(CHECKPOINT_ROOT / "best.pt", map_location=DEVICE)
model.load_state_dict(best_payload["model"])
model.eval()

def collect_predictions(loader):
    predictions, targets, masks = [], [], []
    with torch.no_grad():
        for batch in loader:
            batch = move_batch(batch)
            with torch.cuda.amp.autocast(dtype=amp_dtype, enabled=use_amp):
                prediction, _ = model(batch["inner"], batch["outer"], batch["track"], batch["env"])
            predictions.append((prediction * TARGET_SCALE).float().cpu().numpy())
            targets.append(batch["target"].float().cpu().numpy())
            masks.append(batch["target_mask"].float().cpu().numpy())
    return np.concatenate(predictions), np.concatenate(targets), np.concatenate(masks)

val_pred, val_target, val_mask = collect_predictions(valid_loader)
test_pred, test_target, test_mask = collect_predictions(test_loader) if len(test_idx) else (None, None, None)

def report_metrics(prediction, target, mask):
    metric = {}
    pred_track = np.cumsum(prediction[..., :2], axis=1)
    true_track = np.cumsum(target[..., :2], axis=1)
    metric["track_error_km"] = float(np.sqrt(((pred_track - true_track) ** 2).sum(axis=-1)).mean())
    for index, name in [(2, "vmax_mae_kt"), (3, "pressure_mae_hpa"), (4, "rmw_mae_km")]:
        valid = mask[..., index] > 0.5
        metric[name] = float(np.abs(prediction[..., index][valid] - target[..., index][valid]).mean()) if valid.any() else None
    radius_mask = mask[..., 5:17] > 0.5
    radius_error = np.abs(prediction[..., 5:17] - target[..., 5:17])
    metric["radius_mae_km"] = float(radius_error[radius_mask].mean()) if radius_mask.any() else None
    return metric

val_metrics = report_metrics(val_pred, val_target, val_mask)
print("Validation metrics:", json.dumps(val_metrics, indent=2))
if test_pred is not None:
    print("Test metrics:", json.dumps(report_metrics(test_pred, test_target, test_mask), indent=2))

# Use the complete always-observed subset for a first covariance residual matrix.
complete_indices = [0, 1, 2, 3]
residuals = (val_pred[..., complete_indices] - val_target[..., complete_indices]).reshape(len(val_pred), -1)
np.save(ARTIFACT_ROOT / "validation_residuals_track_wind_pressure.npy", residuals)
np.savez_compressed(
    ARTIFACT_ROOT / "validation_predictions.npz",
    pred=val_pred, target=val_target, mask=val_mask,
)
print("Saved residual matrix:", residuals.shape)
print("Saved validation predictions to Drive.")


## Optional covariance post-processing

Copy covariance_denoise.py into the Colab working directory or upload it with the notebook, then run it on the saved validation residuals. Estimate a and beta on training storms and keep validation storms untouched for evaluation.

Do not report covariance improvement from the synthetic demo as forecast improvement. The actual comparison requires held-out storm residuals and CRPS, energy score, spread-skill, and coverage metrics.


In [ ]:
COVARIANCE_SCRIPT = Path("/content/covariance_denoise.py")
print("Residual file:", ARTIFACT_ROOT / "validation_residuals_track_wind_pressure.npy")
print("To run after uploading covariance_denoise.py:")
print(
    f"python {COVARIANCE_SCRIPT} "
    f"--residuals {ARTIFACT_ROOT / 'validation_residuals_track_wind_pressure.npy'} "
    f"--a 2.0 --beta 0.25 "
    f"--output {ARTIFACT_ROOT / 'covariance_results.npz'}"
)


## Final Drive manifest

The run directory now contains the dataset cache, processed windows, checkpoints, loss plot, validation predictions, and residual matrix.

Before using the model, verify that the ERA5 frames were real, correctly normalized, time-aligned, and created without future information.


In [ ]:
manifest = {
    "run_dir": str(DRIVE_ROOT),
    "best_checkpoint": str(CHECKPOINT_ROOT / "best.pt"),
    "last_checkpoint": str(CHECKPOINT_ROOT / "last.pt"),
    "window_file": str(WINDOW_FILE),
    "loss_plot": str(ARTIFACT_ROOT / "loss_curve.png"),
    "validation_predictions": str(ARTIFACT_ROOT / "validation_predictions.npz"),
    "residuals": str(ARTIFACT_ROOT / "validation_residuals_track_wind_pressure.npy"),
    "config": CONFIG,
    "model_parameters": int(sum(p.numel() for p in model.parameters() if p.requires_grad)),
    "validation_metrics": val_metrics,
    "data_source": "IBTrACS v04r01 WP plus CDS ERA5 pressure and single levels",
    "warning": "Experimental research output; not an operational typhoon warning system.",
}
(DRIVE_ROOT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))
